# Databricks Host and Token
### The Personal Access Token is used to authenticate your SDK calls. It's a secret key, so treat it like a password!
1. In your Databricks workspace, click your username in the top-right corner.
2. Select Settings.
3. Click the Developer tab.
4. Next to Access tokens, click Manage.
5. Click the Generate new token button.
6. Enter a Comment (e.g., "SDK Access") and set a Lifetime for the token (the recommended practice is to set an expiration date).
7. Click Generate.
8. Immediately copy the displayed token. This is the only time Databricks will show you the token. If you lose it, you'll have to generate a new one.
9. Add them to the databricksconfig file generated by the cli install.

In [1]:
from uuid import uuid4
import os
import pandas as pd
from databricks.sdk import WorkspaceClient

# https://github.com/AgDMALabs-Public/ag-vision-dataops
from ag_vision.rover.ingest import RoverDataIngest

# Connect to Roboflow 

In [2]:
w = WorkspaceClient(profile='agpile')  # set your profile
w.config.host

'https://dbc-0f3d94f2-e27b.cloud.databricks.com'

# Define the local and Databricks Variables

In [7]:
SCAN_DATE = '11/25/2025'

YEAR = 2025
COUNTRY = 'IND' # Three letter coutry code.
CROP = 'maize'
CROP_GROWTH_STAGE = 'VT'
TIME_OF_YEAR = 'spring'

TRIAL_NAME = 'BETS'
SITE = 'TNAU-Coimbatore'
FIELD = 'West_Field_03'
LOCATION = 'Section_B'

ROVER_LOCAL_PATH = '/Users/danielwilliams/Documents/Field Data/rover_upload_test'
DB_PROJECT_DIR = '/Volumes/ue1_prod_catalog_119738067017277/tier1_raw/data'

raw_image_dir = f'{ROVER_LOCAL_PATH}/raw_data/'
stiched_image_dir = f'{ROVER_LOCAL_PATH}/stiched_data/'
plot_image_dir = f'{ROVER_LOCAL_PATH}/plot_data/'

plot_boundary_key = f"{ROVER_LOCAL_PATH}/plot_boundary.geojson"
metadata_key = f"{ROVER_LOCAL_PATH}/rover_details.json"

CAMERA = 'rgb'
SCAN_DATE = '1/2/2026'
STICHED_DATE = '1/2/2026'
STICHING_METHOD = 'gemini'
PLOT_CROP_DATE = '1/2/2026'

# Define the Metadata

In [8]:
rover_metadata = {
    "id": str(uuid4()),
    "name": "stand_count",  # this will be the mission name in the folder path
    "task": "rover_data_collection",
    "location": {
        "site": SITE.lower(),  # Try to keep these standard ORG-Site EX CIAT-Cali, or CIAT-Arusha....
        "field": FIELD.lower(),
        # what is the name of the field that the trial was run on? The field and location can be the same name if there is no difference.
        "location": LOCATION.lower()
        # The location corresponds to a specific field book. If data is in EBS match the location name.
    },
    "trialProperties": {
        "name": TRIAL_NAME  # What is the name of the trial, a trial usually has multiple locations.
    },
    "rover_acquisition_properties": {
        "date": SCAN_DATE,
        "rover_make": "NewCo",
        "rover_model": "Rover-1",
        "camera_make": "sony",
        "camera_model": "MX-1000",
        "cameraHeight": 45.5,
        "horizontalOverlapPercentage": 75.0,
        "verticalOverlapPercentage": 70.0,
        "gpsQuality": "RTK Fixed"
    },
    "agronomic_properties": {
        "crop_type": CROP,  # Required
        "growth_stage": CROP_GROWTH_STAGE,  # optional
        "soil_color": None,
        "weed_pressure": None,
        "irrigation_level": None,
        "tillage_type": None,
        "fertilizer_level": None
    }
}

# List out the Raw Images.

In [9]:
raw_files = os.listdir(raw_image_dir)
raw_files = [raw_image_dir + x for x in raw_files]

print(f"there are {len(raw_files)} raw files in the directory")

there are 15 raw files in the directory


D# List out the Stiched Images

In [10]:
stiched_files = os.listdir(stiched_image_dir)
stiched_files = [stiched_image_dir + x for x in stiched_files]

print(f"there are {len(stiched_files)} raw files in the directory")

stiched_df = pd.DataFrame({'src_path': stiched_files})
stiched_df['camera'] = CAMERA
stiched_df['method'] = STICHING_METHOD
stiched_df['stiched_date'] = STICHED_DATE

there are 15 raw files in the directory


# List out the plot Images

In [11]:
plot_files = os.listdir(plot_image_dir)
plot_files = [plot_image_dir + x for x in plot_files]

print(f"there are {len(plot_files)} raw files in the directory")

plot_df = pd.DataFrame({'src_path': plot_files})
plot_df['camera'] = CAMERA
plot_df['file_generation_datetime'] = PLOT_CROP_DATE

there are 15 raw files in the directory


# Logic to add plot ID to the plot_df

In [12]:
# This will need to be custom on how the data is stored.
plot_df['plot_id'] = plot_df['src_path'].apply(lambda x: os.path.splitext(os.path.basename(x))[0])

# Start the Ingest

In [13]:
ingest = RoverDataIngest(platform='local',  # DONT CHANGE THIS
                         cloud_bucket=DB_PROJECT_DIR,
                         cloud_client=w,  # should not need to change.
                         scan_date=SCAN_DATE,
                         plot_boundary_key=plot_boundary_key,
                         scan_metadata_key=metadata_key)


In [14]:
ingest.load_metadata_from_dict(metadata_dict=rover_metadata)

# season is needed to generate the mission dir, This has lots of Validation and will throw assert errors.
ingest.add_season_code_to_metadata(year=YEAR,
                                   country=COUNTRY,
                                   crop=CROP,
                                   time_of_year=TIME_OF_YEAR)

# This is the main dir where all the data will be stored.
ingest.generate_rover_mission_dir_path()

In [15]:
# Save the Metadata
ingest.save_metadata_to_json_local()
ingest.upload_metadata_to_db()

Uploading: 100%|██████████| 1.39k/1.39k [00:01<00:00, 988B/s]  


In [16]:
ingest.upload_scan_plot_boundary_to_db()

Saving to /Volumes/ue1_prod_catalog_119738067017277/tier1_raw/data/tnau-coimbatore/bets/2025:ind:maize:spring/west_field_03/section_b/rover/stand_count/field_data/plot_boundary.geojson


Uploading: 100%|██████████| 190k/190k [00:00<00:00, 327kB/s]


In [17]:
ingest.rover_mission_dir

'/Volumes/ue1_prod_catalog_119738067017277/tier1_raw/data/tnau-coimbatore/bets/2025:ind:maize:spring/west_field_03/section_b/rover/stand_count'

# Ingest the Raw Data

In [18]:
# This will be a list of all the raw files from the flight eg: nav, bin, tif, and jpg files.
ingest.generate_raw_ingest_df(file_list=raw_files,
                              camera=CAMERA)


In [19]:
ingest.generate_raw_image_dst_path_name()

In [20]:
ingest.upload_raw_scan_data_to_db()

0it [00:00, ?it/s]
Uploading:   0%|          | 0.00/7.43M [00:00<?, ?B/s]
Uploading:   3%|▎         | 240k/7.08M [00:00<00:04, 1.67MB/s]
Uploading:   9%|▊         | 624k/7.08M [00:00<00:02, 2.71MB/s]
Uploading:  23%|██▎       | 1.64M/7.08M [00:00<00:00, 6.00MB/s]
Uploading:  40%|███▉      | 2.83M/7.08M [00:00<00:00, 8.08MB/s]
Uploading:  51%|█████▏    | 3.64M/7.08M [00:00<00:00, 5.89MB/s]
Uploading:  61%|██████    | 4.30M/7.08M [00:00<00:00, 5.40MB/s]
Uploading:  69%|██████▉   | 4.88M/7.08M [00:00<00:00, 5.03MB/s]
Uploading:  76%|███████▋  | 5.41M/7.08M [00:02<00:01, 1.12MB/s]
Uploading:  82%|████████▏ | 5.78M/7.08M [00:03<00:01, 875kB/s] 
Uploading:  85%|████████▌ | 6.05M/7.08M [00:04<00:01, 711kB/s]
Uploading:  88%|████████▊ | 6.25M/7.08M [00:04<00:01, 658kB/s]
Uploading:  90%|█████████ | 6.41M/7.08M [00:04<00:01, 637kB/s]
Uploading:  92%|█████████▏| 6.53M/7.08M [00:05<00:00, 646kB/s]
Uploading:  94%|█████████▍| 6.64M/7.08M [00:05<00:00, 645kB/s]
Uploading:  95%|█████████▌| 6.73M/7.0

# Ingest the Stiched Data

In [21]:
ingest.generate_stiched_ingest_df(df=stiched_df,
                                  generate_uuid=True) # This will give the files new UUIDS for the image names. Set to False if you don't want to change it.

In [22]:
ingest.generate_stiched_image_dst_path_name()

In [23]:
ingest.upload_stiched_images_to_db()

0it [00:00, ?it/s]
Uploading:   0%|          | 0.00/7.43M [00:00<?, ?B/s]
Uploading:   0%|          | 16.0k/7.08M [00:00<02:41, 45.7kB/s]
Uploading:   2%|▏         | 160k/7.08M [00:00<00:20, 359kB/s]  
Uploading:   3%|▎         | 208k/7.08M [00:00<00:19, 370kB/s]
Uploading:   4%|▎         | 256k/7.08M [00:00<00:17, 399kB/s]
Uploading:   8%|▊         | 544k/7.08M [00:00<00:06, 1.06MB/s]
Uploading:  11%|█         | 784k/7.08M [00:01<00:05, 1.26MB/s]
Uploading:  26%|██▌       | 1.81M/7.08M [00:01<00:01, 3.71MB/s]
Uploading:  38%|███▊      | 2.69M/7.08M [00:01<00:00, 5.15MB/s]
Uploading:  53%|█████▎    | 3.75M/7.08M [00:01<00:00, 6.72MB/s]
Uploading:  64%|██████▍   | 4.53M/7.08M [00:01<00:00, 7.13MB/s]
Uploading:  76%|███████▋  | 5.41M/7.08M [00:01<00:00, 7.62MB/s]
Uploading:  87%|████████▋ | 6.17M/7.08M [00:01<00:00, 7.15MB/s]
Uploading: 100%|██████████| 7.08M/7.08M [00:05<00:00, 1.41MB/s]
1it [00:05,  5.27s/it]
Uploading:   0%|          | 0.00/7.26M [00:00<?, ?B/s]
Uploading:   2%|▏     

# Ingest the Plot Images

In [20]:
ingest.generate_plot_ingest_df(df=plot_df,
                               generate_uuid=True) # This will give the files new UUIDS for the image names. Set to False if you dont want to change it.

In [21]:
ingest.generate_plot_image_dst_path_name()

In [22]:
ingest.upload_plot_images_to_db()

0it [00:00, ?it/s]
Uploading:   0%|          | 0.00/7.43M [00:00<?, ?B/s]
Uploading:   8%|▊         | 576k/7.08M [00:00<00:01, 5.03MB/s]
Uploading:  15%|█▍        | 1.05M/7.08M [00:00<00:01, 3.28MB/s]
Uploading:  20%|█▉        | 1.39M/7.08M [00:00<00:02, 2.94MB/s]
Uploading:  24%|██▍       | 1.69M/7.08M [00:00<00:02, 2.76MB/s]
Uploading:  28%|██▊       | 1.97M/7.08M [00:00<00:01, 2.72MB/s]
Uploading:  32%|███▏      | 2.23M/7.08M [00:00<00:01, 2.70MB/s]
Uploading:  35%|███▌      | 2.50M/7.08M [00:00<00:02, 2.39MB/s]
Uploading:  39%|███▉      | 2.78M/7.08M [00:01<00:01, 2.48MB/s]
Uploading:  43%|████▎     | 3.03M/7.08M [00:01<00:02, 2.12MB/s]
Uploading:  46%|████▌     | 3.25M/7.08M [00:01<00:01, 2.04MB/s]
Uploading:  50%|████▉     | 3.53M/7.08M [00:01<00:01, 2.23MB/s]
Uploading:  53%|█████▎    | 3.78M/7.08M [00:01<00:01, 2.30MB/s]
Uploading:  57%|█████▋    | 4.03M/7.08M [00:01<00:01, 2.35MB/s]
Uploading:  61%|██████    | 4.31M/7.08M [00:01<00:01, 2.45MB/s]
Uploading:  64%|██████▍   | 4.5